# 05 — Langfuse v4 trace contract (offline)

The-Mailroom filters `MAILROOM_TRACE_NAMES=document-pipeline` and
`MAILROOM_TRACE_TAGS=mailroom`. Sandbox evals write that contract even
when Langfuse is down: `OBSERVABILITY_PROVIDER=none` makes the context
managers no-ops, which is what pytest and these notebooks use.

Canonical code: [`src/mailroom_sandbox/eval/tracing.py`](../src/mailroom_sandbox/eval/tracing.py),
[`docs/tracing.md`](../docs/tracing.md).


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Family constants


In [ ]:
from mailroom_sandbox.eval import tracing

print("root name :", tracing.PIPELINE_TRACE)
print("backend  :", tracing.tracing_backend())
print("env      :", tracing.observability_environment())
print("tags     :", tracing.default_tags("source-fixtures", "agent-sorter"))
print("session  :", tracing.session_id_for("sorter"))
print()
print("NODE_OBSERVATION_TYPES:")
for name, kind in tracing.NODE_OBSERVATION_TYPES.items():
    print(f"  {name:28s} {kind}")


## Public ground truth (never dump `expected_fields`)

Traces may carry `expected_hf_class`, `expected_doc_class`,
`expected_subclass`, `expected`. Extraction gold stays off the wire.


In [ ]:
from mailroom_sandbox.datasets import load_manifest, parse_expected_fields

row = next(r for r in load_manifest() if parse_expected_fields(r))
public = tracing.public_ground_truth(row)
print("row keys   :", sorted(row.keys()))
print("public GT  :", public)
assert "expected_fields" not in public
assert parse_expected_fields(row), "fixture must have extraction gold in the CSV, not on the trace"
print("expected_fields stay in the CSV / experiment log, not on the Langfuse input")


## No-op root chain + child observation + export bookmark


In [ ]:
with tracing.document_pipeline_trace(
    seed="nb05-demo",
    session_id="sandbox-notebook-demo",
    input={"filename": "sample_msa.txt", "matter_id": "SANDBOX-nb05"},
    metadata={"pipeline": "mailroom", "source": "sandbox-notebook", "attempt": 1},
    tags=tracing.default_tags("source-notebook"),
):
    with tracing.child_observation(
        "classify-document",
        as_type=tracing.observation_type_for("classify-document"),
        input=public,
    ) as span:
        span.update(output={"doc_type": "contract", "confidence": 0.97})
    with tracing.child_observation(
        "extract-fields",
        as_type="agent",
        input={"doc_type": "contract"},
    ):
        pass

tracing.emit_langfuse_score("class_correct", 1.0)
tracing.flush_traces()
exported = tracing.export_traces()
print("export path:", exported)
print("last ids   :", tracing.last_trace_ids())
print()
print("The-Mailroom needs Langfuse up (`sandbox up`) plus:")
print("  MAILROOM_TRACE_NAMES=document-pipeline")
print("  MAILROOM_TRACE_TAGS=mailroom")
print("  MAILROOM_TRACE_ENVIRONMENTS=mock,pilot")
print("Phoenix is an optional sidecar; The-Mailroom cannot plot Phoenix spans.")
